# TIDE-GR **v2** on KuaiRand-Pure (Colab T4) — branch `v2`

This notebook runs **version 2** side by side with v1. It never touches the v1 results:
* code: the `v2` branch of the repository (v1 stays on `main`);
* results: a **separate** Drive folder (`TIDE_GR_v2_runs`) and a **separate** Google Sheet.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Resuming works exactly like v1: if the session ends, open this notebook on any account that has the
`TIDE_GR_v2_runs` folder in "My Drive" (shared folder → *Organize → Add shortcut to My Drive*) and run the
cells again (skip the quick check). Finished runs are skipped; the unfinished run continues from its checkpoint.


In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-eec38d7c-aea3-ad5d-9b6e-1a032ae45371)


In [2]:
!nvidia-smi
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))


Mon Nov 23 17:03:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Settings — the v2 runs folder must be different from the v1 folder

In [ ]:
RUNS_ROOT = "/content/drive/MyDrive/TIDE_GR_v2_runs"   # <- v2 folder (NOT the v1 TIDE_GR_runs folder)
REPO = "https://github.com/S-V-Kartheek/TIDE_GR.git"
BRANCH = "v2"
MAX_HOURS = 11.0    # stop cleanly before the Colab limit; re-run to continue
SHEET = "TIDE-GR v2 results"   # <- paste the URL of a NEW Google Sheet for v2 (do not reuse the v1 sheet)
SA_KEY = None
assert "TIDE_GR_runs" != RUNS_ROOT.rstrip("/").split("/")[-1], "use a separate folder for v2"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(RUNS_ROOT, exist_ok=True)
print("runs folder:", RUNS_ROOT, "contents:", os.listdir(RUNS_ROOT))

## 2. Get / update the code
If the repository is private, add a Colab secret named `GITHUB_TOKEN` (key icon on the left) with a GitHub token that can read the repo.

In [ ]:
import os, subprocess
url = REPO
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    if tok:
        url = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists('/content/TIDE_GR'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/TIDE_GR'], check=True)
else:
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/TIDE_GR', 'pull', '-q', 'origin', BRANCH], check=True)
%cd /content/TIDE_GR
!git branch --show-current
!git log -1 --oneline
!pip install -q -r requirements.txt


## 3. Quick check (optional, ≈15–30 min, run once)
Builds the dataset in the v2 folder and checks metrics + resume. You can skip it if you are short of time.

In [ ]:
!python -m tidegr.quick_check --data data/kuairand.yaml --runs_root "$RUNS_ROOT"

## 4. Full v2 plan (resumable — just run this cell again after any interruption)
Order: TIDE-GR v2 + main baselines (seed 0) → more seeds → v2 ablations (RL controller first) → remaining baselines.
The run called `tidegr` is TIDE-GR **v2**; `tidegr_v1` is the unchanged v1 model re-run for comparison.

In [ ]:
!python -m tidegr.run_all --plan plans/kuairand_v2.yaml --runs_root "$RUNS_ROOT" --max_hours $MAX_HOURS

## 5. Progress

In [ ]:
import json
st = json.load(open(f"{RUNS_ROOT}/kuairand/status.json"))
for k, v in st.items():
    print(f"{k:35s} {v.get('state'):8s} {v.get('ndcg@20') or ''}")

## 6. Results → Drive + Google Sheet + paper figures (run any time; re-run after new runs finish)
* Tables: `<RUNS_ROOT>/kuairand/results/` (summary.csv, table.md, significance.csv)
* Figures (PDF for LaTeX + PNG): `<RUNS_ROOT>/kuairand/results/figures/`
* Google Sheet tabs: `kuairand_table`, `kuairand_runs`, `kuairand_blocks`, `kuairand_significance`, `kuairand_status`, `log`
The first time, Google asks you to allow access (Sheets + Drive). Each run re-writes the tabs from the files on Drive.

In [ ]:
!python -m tidegr.report --runs_root "$RUNS_ROOT" --dataset kuairand

In [ ]:
import sys; sys.path.insert(0, '/content/TIDE_GR')
from tidegr.sheets import sync
sync(RUNS_ROOT, 'kuairand', SHEET, sa_key=SA_KEY)

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f'{RUNS_ROOT}/kuairand/results/figures/*.png')):
    print(p); display(Image(p, width=520))